# Lab 09-03 — Tree-vs-flat retrieval eval (RAPTOR step 3)

**Track 09 · RAPTOR** — replacing flat chunk lists with a summary tree.

Labs 01-02 built the RAPTOR tree. This lab asks the question that decides whether the tree is worth its LLM calls: does *tree retrieval* find the right chunks, and how does it compare with the flat baseline it replaces?

This notebook is **self-contained**: it imports LangChain (the BGE embedder and the local Ollama LLM), pandas, and scikit-learn directly — no repo component library. The RAPTOR machinery the repo ships as `src/tools/raptor.py` — recursive GMM clustering, LLM summarization, tree build, and the collapsed-tree traversal — is hand-rolled right here, cell by cell, which is exactly how that shared component works underneath.

```text
24 chunks (rag-mini-wikipedia) + 3 test questions (deterministic heads)
  -> BGE embeddings (BAAI/bge-base-en-v1.5, local, CPU)
  -> RAPTOR tree build (recursive GMM + ChatOllama summaries)
  -> flat baseline  : cosine-rank all 24 passages, take top-4
  -> tree retrieval : walk summaries top-4 at a time, collapse to leaf chunks
  -> gold-answer words surfaced in the top-1 passage, per method
  -> verification gate
```

* **Flat baseline** — embed the question and cosine-rank every one of the N=24 passages directly; take the top 4. This is standard vector search: O(N) similarity computations, no summaries.
* **Tree retrieval** — walk the tree instead: at each level, embed the question, keep the `top_k` most similar children, descend into them, and finally return the best *leaf* chunks (`collapse=True`). Each step only compares against a handful of summaries, so a query that matches a topic broad enough to be summarized can jump straight to the right cluster without scanning every chunk.

We evaluate both methods on 3 yes/no questions from the corpus test set and report, per question, the top passage each method surfaces and whether that passage contains the gold answer's words (an `answer_contains`-style normalized substring check). The hit rates are the lesson — they are printed but not hard-required by the verification gate; the gate checks only that both methods return well-formed retrievals.


## Setup

Two prerequisites must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` + `test.parquet`, already fetched by the repo's manifest-verified fetchers.
- **Ollama serving `qwen2.5-coder:7b`** — the fully local LLM that writes the summaries during the tree build (`ollama pull qwen2.5-coder:7b`, then `ollama serve`). Only the build uses it; retrieval and the flat baseline are LLM-free. No API key.

No repo imports are needed: everything this notebook uses comes from `langchain-huggingface`, `langchain-ollama`, `pandas`, `numpy`, and `scikit-learn`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   langchain-huggingface -> HuggingFaceEmbeddings (BGE backend)
#   langchain-ollama      -> ChatOllama (local qwen2.5-coder:7b summaries)
#   pandas                -> read the rag-mini-wikipedia parquets
#   scikit-learn, numpy   -> GaussianMixture clustering (hand-rolled RAPTOR)
%pip install -q langchain-huggingface langchain-ollama pandas scikit-learn


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
import urllib.request
from pathlib import Path

# LangChain + pandas + scikit-learn — the only libraries this notebook
# needs. Nothing is imported from the repo's src/ component library.
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_ollama import ChatOllama  # noqa: E402
from sklearn.mixture import GaussianMixture  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `PASSAGES_PATH` and `TEST_PATH` point at the corpus and test set already on disk; `N_PASSAGES = 24` is the corpus both methods search (deterministic head); `N_QUESTIONS = 3` is the deterministic head of the test set; `TOP_K = 4` is the retrieved-chunks-per-method-per-question count; `MAX_CLUSTER_SIZE = 8` caps the summary nodes in the tree; the BGE constants pin the embedder to the local model on CPU; `PREVIEW` truncates the passage previews the demo prints.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 24  # corpus both methods search; tree build costs ~1 LLM call/cluster
N_QUESTIONS = 3  # deterministic head of the test set
TOP_K = 4  # retrieved chunks per method per question
MAX_CLUSTER_SIZE = 8  # max chunks per summary node in the tree
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM
OLLAMA_MODEL = "qwen2.5-coder:7b"  # local LLM, same default the repo uses
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_TEMPERATURE = 0.0  # deterministic summaries
PREVIEW = 100  # characters of each top passage to print


## 2. Load — first N passages + first N test questions of rag-mini-wikipedia

`load_passages` reads the first `n` rows of `passages.parquet` (stripped, file order); `load_questions` reads the first `n` rows of `test.parquet` as `{"question", "answer"}` pairs. Both are deterministic heads, exactly the slices the lab script uses.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages + first N test questions of rag-mini-wikipedia
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` ``{"question", "answer"}`` pairs (deterministic)."""
    df = pd.read_parquet(path)
    return [
        {
            "question": str(row["question"]).strip(),
            "answer": str(row["answer"]).strip(),
        }
        for _, row in df.head(n).iterrows()
    ]


## 3. Experiment — build the tree once, then compare flat vs tree retrieval

The whole pipeline is built inline. **Index** — the RAPTOR tree build reuses the hand-rolled pieces from labs 01-02: recursive 2-component full-covariance `GaussianMixture` clustering, `ChatOllama` summarization (with the deterministic `_StubLLM` fallback and the same `[RUN]`/`[SKIP]` contract), and the recursive `build_tree` ending at a single root. **Retrieval** — `flat_top_k` cosine-ranks all `N_PASSAGES` passage vectors against the question (plain vector search, O(N)); `retrieve` walks the tree instead: at each level it embeds the frontier node texts, keeps the `top_k` most similar to the question, descends into their children, and with `collapse=True` finally ranks the leaf chunks and returns the best `TOP_K`. Both use the same `_cosine` score, so the comparison is apples-to-apples. `answer_contains` is the normalized substring check (are the gold answer's words in the top-1 passage?).


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — build the tree once, then compare flat vs tree retrieval
# --------------------------------------------------------------------------
SUMMARY_PROMPT = (
    "Summarize the following passages in 2-3 sentences, keeping key facts "
    "and names."
)


def _cosine(a: list[float], b: list[float]) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def _split_indices(
    index_set: list[int],
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int,
) -> list[list[int]]:
    """Recursively split ``index_set`` into clusters of at most ``max_cluster_size``."""
    if len(index_set) <= max_cluster_size:
        return [sorted(index_set)]
    matrix = np.asarray([embeddings[i] for i in index_set], dtype=float)
    try:
        labels = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=seed,
        ).fit_predict(matrix)
    except (ValueError, np.linalg.LinAlgError):
        # Fallback: keep the whole set as one cluster on fit failure
        # (ill-defined covariance, n_components > n_samples) so coverage holds.
        return [sorted(index_set)]

    groups: dict[int, list[int]] = {}
    for idx, label in zip(index_set, labels):
        groups.setdefault(int(label), []).append(idx)
    if len(groups) < 2:  # degenerate split: every point in one component
        return [sorted(index_set)]

    return [
        cluster
        for group in groups.values()
        for cluster in _split_indices(group, embeddings, max_cluster_size, seed)
    ]


def cluster_embeddings(
    embeddings: list[list[float]],
    max_cluster_size: int,
    seed: int = 42,
) -> list[list[int]]:
    """Partition chunk indices into GMM clusters of at most ``max_cluster_size``."""
    if not embeddings:
        return []
    return _split_indices(
        list(range(len(embeddings))), embeddings, max_cluster_size, seed
    )


def _summarize(llm, texts: list[str]) -> str:
    """One LLM call: compress ``texts`` into 2-3 sentences of prose."""
    joined = "\n".join(f"- {text}" for text in texts)
    out = llm.invoke(f"{SUMMARY_PROMPT}\n\n{joined}")
    if hasattr(out, "content"):  # ChatOllama returns an AIMessage
        out = out.content
    return str(out).strip()


class _StubLLM:
    """Deterministic summarizer used only when Ollama is unreachable."""

    def invoke(self, prompt: str) -> str:
        lines = [line[2:] for line in prompt.splitlines() if line.startswith("- ")]
        return " ".join(line[:24] for line in lines)[:200] or "summary"


def _ollama_reachable(url: str = OLLAMA_BASE_URL, timeout: float = 2.0) -> bool:
    """True if the local Ollama server answers /api/tags."""
    try:
        with urllib.request.urlopen(f"{url}/api/tags", timeout=timeout) as resp:
            return resp.status == 200
    except Exception:
        return False


def build_tree(
    chunks: list[str],
    embedder,
    llm,
    max_cluster_size: int = 8,
    progress=None,
) -> dict:
    """Build the RAPTOR tree: leaves are chunks, parents are LLM summaries.

    Returns ``{"tree", "levels", "leaves", "llm_calls"}``. ``tree`` is a
    nested dict with shape ``{"text", "chunk_ids", "children", "level"}`` —
    level-0 nodes are leaves (one chunk each, empty ``children``); higher
    levels are summaries whose ``chunk_ids`` cover the whole subtree. Every
    chunk appears in exactly one leaf.
    """
    nodes = [
        {"text": chunks[i], "chunk_ids": [i], "children": [], "level": 0}
        for i in range(len(chunks))
    ]
    llm_calls = 0
    level = 0
    while len(nodes) > 1:
        level += 1
        vectors = embedder.embed_documents([node["text"] for node in nodes])
        clusters = cluster_embeddings(vectors, max_cluster_size)
        if len(clusters) == len(nodes):
            # No merge happened (degenerate GMM split): collapse to a single
            # root so the recursion always terminates.
            clusters = [list(range(len(nodes)))]
        parents: list[dict] = []
        for cluster in clusters:
            members = [nodes[i] for i in cluster]
            parents.append(
                {
                    "text": _summarize(llm, [node["text"] for node in members]),
                    "chunk_ids": sorted(
                        cid for node in members for cid in node["chunk_ids"]
                    ),
                    "children": members,
                    "level": level,
                }
            )
            llm_calls += 1
        nodes = parents
        if progress is not None:
            progress(level, len(nodes))

    if not nodes:  # empty chunk list: keep a harmless empty root
        nodes = [{"text": "", "chunk_ids": [], "children": [], "level": 0}]
    root = nodes[0]
    return {
        "tree": root,
        "levels": level,
        "leaves": len(_collect_chunk_ids(root)),
        "llm_calls": llm_calls,
    }


def _collect_chunk_ids(node: dict) -> list[int]:
    """All chunk ids under ``node`` (leaf walk) — useful for gate checks."""
    if not node["children"]:
        return list(node["chunk_ids"])
    ids: list[int] = []
    for child in node["children"]:
        ids.extend(_collect_chunk_ids(child))
    return ids


def retrieve(
    tree: dict,
    question: str,
    embedder,
    top_k: int = 4,
    collapse: bool = True,
) -> dict:
    """Traverse the tree for ``question``.

    At each level the ``top_k`` children most similar to the question (by
    cosine) are kept and their children searched next. With ``collapse=True``
    the traversal keeps descending to the leaves and returns the best leaf
    chunk texts. Returns ``{"texts", "ids", "scores", "path"}``.
    """
    question_vec = embedder.embed_documents([question])[0]
    path: list[list[str]] = []
    frontier = [tree]
    last_selected = [tree]
    last_scores = [1.0]
    while frontier and frontier[0].get("children"):
        vectors = embedder.embed_documents([node["text"] for node in frontier])
        ranked = sorted(
            enumerate(frontier),
            key=lambda pair: _cosine(question_vec, vectors[pair[0]]),
            reverse=True,
        )[:top_k]
        last_selected = [frontier[i] for i, _ in ranked]
        last_scores = [_cosine(question_vec, vectors[i]) for i, _ in ranked]
        path.append([node["text"] for node in last_selected])
        frontier = [
            child for node in last_selected for child in node["children"]
        ]

    if not collapse:
        return {
            "texts": [node["text"] for node in last_selected],
            "ids": [
                cid for node in last_selected for cid in node["chunk_ids"]
            ],
            "scores": [round(score, 3) for score in last_scores],
            "path": path,
        }

    leaves = [node for node in frontier if not node.get("children")] or [tree]
    vectors = embedder.embed_documents([node["text"] for node in leaves])
    ranked = sorted(
        enumerate(leaves),
        key=lambda pair: _cosine(question_vec, vectors[pair[0]]),
        reverse=True,
    )[:top_k]
    return {
        "texts": [leaves[i]["text"] for i, _ in ranked],
        "ids": [leaves[i]["chunk_ids"][0] for i, _ in ranked],
        "scores": [
            round(_cosine(question_vec, vectors[i]), 3) for i, _ in ranked
        ],
        "path": path,
    }


def flat_top_k(
    passages: list[str],
    passage_vecs: list[list[float]],
    question_vec: list[float],
    top_k: int,
) -> dict:
    """Rank all passages against the question; return the ``top_k``."""
    ranked = sorted(
        range(len(passages)),
        key=lambda i: _cosine(question_vec, passage_vecs[i]),
        reverse=True,
    )[:top_k]
    return {
        "texts": [passages[i] for i in ranked],
        "ids": list(ranked),
        "scores": [round(_cosine(question_vec, passage_vecs[i]), 3)
                   for i in ranked],
    }


def answer_contains(gold: str, text: str) -> bool:
    """Normalized substring check: are the gold answer's words in ``text``?"""
    return gold.strip().lower() in text.strip().lower()


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, N_QUESTIONS)
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": BGE_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )

    # Run/skip contract: real ChatOllama when the server answers, deterministic
    # stub otherwise — the gate passes either way.
    if _ollama_reachable():
        print(f"[RUN] Ollama LLM {OLLAMA_MODEL} reachable at {OLLAMA_BASE_URL} "
              f"(temperature {OLLAMA_TEMPERATURE})")
        llm = ChatOllama(
            model=OLLAMA_MODEL,
            temperature=OLLAMA_TEMPERATURE,
            base_url=OLLAMA_BASE_URL,
        )
        llm_mode = "ollama"
    else:
        print(f"[SKIP] Ollama not reachable at {OLLAMA_BASE_URL} — using "
              "deterministic stub summarizer")
        llm = _StubLLM()
        llm_mode = "stub"

    t_start = time.perf_counter()
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    tree = build_tree(passages, embedder, llm, max_cluster_size=MAX_CLUSTER_SIZE)
    build_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    rows: list[dict] = []
    for q in questions:
        question_vec = embedder.embed_documents([q["question"]])[0]
        flat = flat_top_k(passages, passage_vecs, question_vec, TOP_K)
        tree_ret = retrieve(
            tree["tree"], q["question"], embedder,
            top_k=TOP_K, collapse=True,
        )
        rows.append(
            {
                "question": q["question"],
                "gold": q["answer"],
                "flat": flat,
                "flat_gold_words": answer_contains(q["answer"], flat["texts"][0]),
                "tree": tree_ret,
                "tree_gold_words": answer_contains(q["answer"], tree_ret["texts"][0]),
            }
        )
    query_s = time.perf_counter() - t0
    total_s = time.perf_counter() - t_start

    return {
        "passages": passages,
        "questions": questions,
        "rows": rows,
        "tree": tree,
        "llm_mode": llm_mode,
        "embed_s": embed_s,
        "build_s": build_s,
        "query_s": query_s,
        "total_s": total_s,
        "agg": {
            "questions": len(rows),
            "flat_gold_words": sum(r["flat_gold_words"] for r in rows),
            "tree_gold_words": sum(r["tree_gold_words"] for r in rows),
        },
    }


## 4. Demo — print the comparison

`print_demo(exp)` prints the comparison from four angles: the corpus headline (passages, tree levels/leaves/LLM calls, build vs total time, LLM mode); per question — gold answer, the flat top-1 passage with its id and whether it carries the gold answer's words, the tree top-1 passage with the same check; the aggregated hit counts (flat vs tree, reported not required); then a takeaway explaining the index-time/query-time trade.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the comparison
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-03 — Tree-vs-flat retrieval eval")
    t = exp["tree"]
    print(f"{len(exp['passages'])} passages; tree {t['levels']} levels / "
          f"{t['leaves']} leaves / {t['llm_calls']} LLM calls in "
          f"{exp['build_s']:.1f}s (total {exp['total_s']:.1f}s) "
          f"[llm mode: {exp['llm_mode']}]")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        flat, tree = row["flat"], row["tree"]
        print(f"\nQ{i}: {row['question']}")
        print(f"    gold answer    : {row['gold']}")
        print(f"    flat  top [id {flat['ids'][0]}] "
              f"{flat['texts'][0][:PREVIEW]}")
        print(f"    flat  gold words : {row['flat_gold_words']}")
        print(f"    tree  top [id {tree['ids'][0]}] "
              f"{tree['texts'][0][:PREVIEW]}")
        print(f"    tree  gold words : {row['tree_gold_words']}")

    a = exp["agg"]
    print(f"\n[5] Gold-answer words surfaced in the top-1 passage "
          f"({a['questions']} questions)")
    print(f"    flat : {a['flat_gold_words']}/{a['questions']}")
    print(f"    tree : {a['tree_gold_words']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    The tree trades a few LLM calls at index time for query-time")
    print("    retrieval that only compares against handfuls of summaries.")
    print("    It shines when a question maps to a whole cluster; the flat")
    print("    baseline stays competitive on small corpora like this one, so")
    print("    the right choice depends on corpus size and query breadth.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: per question, both methods return exactly `TOP_K` non-empty texts and `TOP_K` ids, and every id is a valid chunk index in `0..N_PASSAGES`; scores are sane (0.0..1.0) for both methods; and the tree is well-formed (levels >= 1, leaves == `N_PASSAGES`, `llm_calls <= 40`). The gold-answer hit rates are printed as an informational line — reported, not required. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a = exp["agg"]
    t = exp["tree"]

    for i, row in enumerate(exp["rows"], start=1):
        flat, tree = row["flat"], row["tree"]
        checks.append((f"Q{i} flat returns {TOP_K} non-empty texts + "
                       f"{TOP_K} ids",
                       len(flat["texts"]) == TOP_K
                       and len(flat["ids"]) == TOP_K
                       and all(text.strip() for text in flat["texts"])))
        checks.append((f"Q{i} tree returns {TOP_K} non-empty texts + "
                       f"{TOP_K} ids",
                       len(tree["texts"]) == TOP_K
                       and len(tree["ids"]) == TOP_K
                       and all(text.strip() for text in tree["texts"])))
        checks.append((f"Q{i} flat ids are valid chunk indices (0.."
                       f"{len(exp['passages'])})",
                       all(0 <= idx < len(exp["passages"])
                           for idx in flat["ids"])))
        checks.append((f"Q{i} tree ids are valid chunk indices (0.."
                       f"{len(exp['passages'])})",
                       all(0 <= idx < len(exp["passages"])
                           for idx in tree["ids"])))

    checks.append(("scores are sane (0.0..1.0) for both methods",
                   all(0.0 <= score <= 1.0
                       for row in exp["rows"]
                       for score in row["flat"]["scores"] + row["tree"]["scores"])))
    checks.append((f"tree is well-formed (levels >= 1, leaves == "
                   f"{len(exp['passages'])}, llm_calls <= 40)",
                   t["levels"] >= 1
                   and t["leaves"] == len(exp["passages"])
                   and t["llm_calls"] <= 40))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    print(f"  (info) gold-answer words in top-1: flat "
          f"{a['flat_gold_words']}/{a['questions']}, tree "
          f"{a['tree_gold_words']}/{a['questions']} — reported, not required")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes: 24 passages embedded on CPU, the RAPTOR tree built with one `ChatOllama` call per internal node (typically 4-8 calls at temperature 0), then 3 questions through both retrieval methods. If Ollama is down, the stub summarizer keeps the run green and the gate passes. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per question: what each method surfaced as its top-1 passage (flat = raw cosine over all chunks; tree = walked summaries collapsed to leaves), and whether that passage contains the gold answer's words — the hit counts that make the lesson visible.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact and Ollama is serving `qwen2.5-coder:7b`.


In [ ]:
verify_gate(exp)
